In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

train_dataset = datasets.GTSRB(
    root="./data",
    split="train",
    download=True,
    transform=transform
)

test_dataset = datasets.GTSRB(
    root="./data",
    split="test",
    download=True,
    transform=transform
)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

num_classes = 43

100%|██████████| 187M/187M [00:13<00:00, 14.3MB/s]
100%|██████████| 89.0M/89.0M [00:05<00:00, 17.5MB/s]
100%|██████████| 99.6k/99.6k [00:00<00:00, 210kB/s]


In [3]:
def train_one_epoch(model, loader, criterion, optimizer):
    model.train()
    total_loss = 0.0

    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(loader)


def evaluate(model, loader):
    model.eval()
    correct, total = 0, 0

    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, preds = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (preds == labels).sum().item()

    return 100 * correct / total

In [4]:
def get_resnet18():
    model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
    model.fc = nn.Linear(model.fc.in_features, num_classes)
    return model.to(device)

VARIANT 1: Frozen Backbone

In [8]:
model_v1 = get_resnet18()

for param in model_v1.parameters():
    param.requires_grad = False

for param in model_v1.fc.parameters():
    param.requires_grad = True

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model_v1.fc.parameters(), lr=1e-3)

for epoch in range(3):
    loss = train_one_epoch(model_v1, train_loader, criterion, optimizer)
    acc = evaluate(model_v1, test_loader)
    print(f"[V1] Epoch {epoch+1} | Loss: {loss:.4f} | Acc: {acc:.2f}%")

[V1] Epoch 1 | Loss: 1.4213 | Acc: 65.21%
[V1] Epoch 2 | Loss: 0.7199 | Acc: 67.66%
[V1] Epoch 3 | Loss: 0.5658 | Acc: 68.59%


VARIANT 2: Partially Frozen Backbone

In [6]:
model_v2 = get_resnet18()

for param in model_v2.parameters():
    param.requires_grad = False

for param in model_v2.layer4.parameters():
    param.requires_grad = True

for param in model_v2.fc.parameters():
    param.requires_grad = True

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(
    filter(lambda p: p.requires_grad, model_v2.parameters()),
    lr=1e-3
)

for epoch in range(3):
    loss = train_one_epoch(model_v2, train_loader, criterion, optimizer)
    acc = evaluate(model_v2, test_loader)
    print(f"[V2] Epoch {epoch+1} | Loss: {loss:.4f} | Acc: {acc:.2f}%")

[V2] Epoch 1 | Loss: 0.2410 | Acc: 92.83%
[V2] Epoch 2 | Loss: 0.0484 | Acc: 93.52%
[V2] Epoch 3 | Loss: 0.0319 | Acc: 93.86%


VARIANT 3: Fully Trainable Model

In [7]:
model_v3 = get_resnet18()

for param in model_v3.parameters():
    param.requires_grad = True

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model_v3.parameters(), lr=1e-3)

for epoch in range(3):
    loss = train_one_epoch(model_v3, train_loader, criterion, optimizer)
    acc = evaluate(model_v3, test_loader)
    print(f"[V3] Epoch {epoch+1} | Loss: {loss:.4f} | Acc: {acc:.2f}%")

[V3] Epoch 1 | Loss: 0.1913 | Acc: 95.89%
[V3] Epoch 2 | Loss: 0.0411 | Acc: 97.16%
[V3] Epoch 3 | Loss: 0.0270 | Acc: 97.99%
